# Gradients and Directional Derivatives

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.04 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/05_gradients_and_directional_derivatives.ipynb)

---

## 🎯 Learning Objective

Understand the gradient as the direction of steepest ascent, and directional derivatives as the rate of change along any chosen direction.

---

## 📐 Theory

In `02.03` the gradient $\nabla f = (\partial f/\partial x_1, \dots, \partial f/\partial x_n)$
was defined as a bundle of partial derivatives. This notebook asks: what does $\nabla f$ *mean*
geometrically, and how do you compute the rate of change in a direction that isn't one of the
coordinate axes?

### Directional derivatives

The **directional derivative** of $f$ at point $\mathbf{a}$ in the direction of a unit vector
$\hat{\mathbf{u}}$ is the rate of change of $f$ as you move from $\mathbf{a}$ along
$\hat{\mathbf{u}}$:

$$D_{\hat{\mathbf{u}}} f(\mathbf{a}) = \lim_{h\to 0}\frac{f(\mathbf{a}+h\hat{\mathbf{u}}) - f(\mathbf{a})}{h}$$

Partial derivatives are the special case where $\hat{\mathbf{u}}$ is a coordinate axis (e.g.
$\hat{\mathbf{u}}=(1,0,\dots,0)$ gives $\partial f/\partial x_1$). The key computational fact,
provable via the multivariate chain rule from `02.04` (treat $g(h) = f(\mathbf{a}+h\hat{\mathbf{u}})$
and differentiate at $h=0$), is:

$$D_{\hat{\mathbf{u}}} f(\mathbf{a}) = \nabla f(\mathbf{a}) \cdot \hat{\mathbf{u}}$$

The directional derivative in *any* direction is just the dot product of the gradient with
that direction. You never need a new formula per direction — the gradient already contains
enough information to answer the question for every direction at once.

### The gradient points toward steepest ascent

Recall from `01.01` that $\mathbf{a}\cdot\mathbf{b} = \|\mathbf{a}\|\|\mathbf{b}\|\cos\theta$.
Applying this to $D_{\hat{\mathbf{u}}}f(\mathbf{a}) = \nabla f(\mathbf{a})\cdot\hat{\mathbf{u}} =
\|\nabla f(\mathbf{a})\|\cos\theta$ (since $\hat{\mathbf{u}}$ has unit length), where $\theta$
is the angle between $\hat{\mathbf{u}}$ and $\nabla f(\mathbf{a})$. This is maximized exactly
when $\theta=0$ — i.e. when $\hat{\mathbf{u}}$ points in the *same direction* as
$\nabla f(\mathbf{a})$, giving a maximum rate of increase of $\|\nabla f(\mathbf{a})\|$. This
single piece of trigonometry is the entire justification for gradient-based optimization:

- $\nabla f(\mathbf{a})$ is the direction of **steepest ascent** (fastest increase).
- $-\nabla f(\mathbf{a})$ is the direction of **steepest descent** (fastest decrease).
- Moving perpendicular to $\nabla f(\mathbf{a})$ ($\theta=90°$) gives zero instantaneous
  change — this is the tangent direction to the level curve $f=c$ through $\mathbf{a}$,
  connecting back to the contour picture from `02.03`.

### Gradients are perpendicular to level curves

Since moving along a level curve doesn't change $f$, the directional derivative along the
level curve's tangent is $0$, which by $\nabla f\cdot\hat{\mathbf{u}}=0$ means $\nabla f$ is
*perpendicular* to that tangent. So $\nabla f(\mathbf{a})$ always points directly "outward,"
perpendicular to the contour line through $\mathbf{a}$, toward higher values of $f$ — this is
exactly the picture behind every 2D loss-landscape gradient arrow you'll see from here on.

### Why steepest descent, not "any decreasing direction"

Any direction with $\theta \in (90°, 270°)$ decreases $f$ locally, so gradient descent isn't
the *only* way to decrease a loss — but it decreases it *fastest*, per unit step size, which is
exactly the property that makes it a good default choice, revisited rigorously in Module `04`.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

On a contour plot, the gradient arrow at any point is perpendicular to the contour line
through that point and always points toward higher values — plotting gradient arrows across a
grid turns the whole surface's "flow toward the summit" into a single picture.

In [ ]:
# f(x,y) = x^2 + 2*y^2, an elliptical bowl. Gradient field overlaid on contours shows
# every arrow perpendicular to its local contour line, pointing "uphill".
f = lambda x, y: x**2 + 2 * y**2
grad_f = lambda x, y: np.array([2 * x, 4 * y])   # analytic gradient, by the power rule per-term

x_grid = np.linspace(-3, 3, 200)
y_grid = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x_grid, y_grid)
Z = f(X, Y)

# Sparser grid for the quiver arrows so they stay readable
xq, yq = np.meshgrid(np.linspace(-2.5, 2.5, 10), np.linspace(-2.5, 2.5, 10))
Uq, Vq = grad_f(xq, yq)
magnitude = np.sqrt(Uq**2 + Vq**2)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=12, cmap="viridis", alpha=0.6)
ax.quiver(xq, yq, Uq / magnitude, Vq / magnitude, magnitude, cmap="autumn", scale=22)
a_pt = (1.5, 1.0)
ax.plot(*a_pt, 'ko', markersize=8)
g = grad_f(*a_pt)
ax.annotate("", xy=(a_pt[0] + g[0]*0.15, a_pt[1] + g[1]*0.15), xytext=a_pt,
            arrowprops=dict(arrowstyle="->", color="#C44E52", lw=2.5))
ax.set_title("Gradient field: every arrow perpendicular to its contour, pointing uphill")
ax.set_xlabel("x"); ax.set_ylabel("y")
plt.show()

print(f"At point {a_pt}: gradient = {g}, ||gradient|| = {np.linalg.norm(g):.3f}")
print("This is the direction of steepest ascent -- steepest DESCENT is exactly the negative of it.")

## 🐍 Implementation from Scratch

We compute directional derivatives directly from the definition (a 1D limit along a chosen
direction) and confirm the shortcut formula $D_{\hat{\mathbf{u}}}f = \nabla f\cdot\hat{\mathbf{u}}$,
then empirically confirm the gradient direction maximizes the directional derivative by scanning
every possible direction.

In [ ]:
def directional_derivative_numeric(f, point, direction, h=1e-6):
    """D_u f(a) straight from the definition: step h along the unit direction and
    measure the rate of change -- no dot-product shortcut used here."""
    u = np.array(direction) / np.linalg.norm(direction)  # normalize to a unit vector
    point = np.array(point, dtype=float)
    return (f(*(point + h * u)) - f(*(point - h * u))) / (2 * h)

def gradient(f, point, h=1e-6):
    point = np.array(point, dtype=float)
    grad = np.zeros_like(point)
    for i in range(len(point)):
        step = np.zeros_like(point); step[i] = h
        grad[i] = (f(*(point + step)) - f(*(point - step))) / (2 * h)
    return grad

f = lambda x, y: x**2 + 2 * y**2
point = (1.5, 1.0)
grad_at_point = gradient(f, point)

# Check the shortcut formula D_u f = grad(f) . u_hat against 4 different directions
directions = {"+x axis": (1, 0), "+y axis": (0, 1), "diagonal": (1, 1), "gradient itself": tuple(grad_at_point)}
print(f"Gradient at {point}: {np.round(grad_at_point, 4)}\n")
for name, d in directions.items():
    u_hat = np.array(d) / np.linalg.norm(d)
    numeric = directional_derivative_numeric(f, point, d)
    shortcut = grad_at_point @ u_hat
    print(f"{name:16s} u_hat={np.round(u_hat,3)}  D_u(numeric)={numeric:.4f}  grad.u_hat={shortcut:.4f}")

# Scan ALL directions (angle 0 to 360 degrees) and confirm the max directional derivative
# occurs exactly where theta aligns with the gradient's own direction
angles = np.linspace(0, 2*np.pi, 360)
directional_values = [grad_at_point @ np.array([np.cos(t), np.sin(t)]) for t in angles]
best_angle = angles[np.argmax(directional_values)]
gradient_angle = np.arctan2(grad_at_point[1], grad_at_point[0]) % (2*np.pi)
print(f"\nAngle that maximizes D_u f: {np.degrees(best_angle):.1f} deg")
print(f"Angle of the gradient itself:   {np.degrees(gradient_angle):.1f} deg  (should match)")

## 🤖 Why This Matters for AI

Gradient descent's update rule, $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L(\mathbf{w})$,
is a direct application of this notebook's central result: $-\nabla L$ is the direction of
steepest *decrease*, so stepping along it shrinks the loss faster than stepping in any other
direction (for a small enough step size $\eta$, formalized in Module `04`). Directional
derivatives also power **feature attribution**: given a trained model, the directional
derivative of its output along a specific input direction tells you how sensitive the
prediction is to changing the input that way — the basis of saliency and "what-if" input
perturbation analysis used to interpret model decisions. Below, we implement one step of
gradient descent from first principles and verify it against the directional-derivative
argument: stepping along $-\nabla L$ decreases the loss more than stepping the same distance in
any other direction we try.

In [ ]:
# A toy "loss" L(w1, w2) for linear regression on one data point -- verify that stepping
# along -grad(L) beats stepping the same distance in any other direction.
x_data, y_data = 2.0, 5.0  # one training example: input x, target y
loss = lambda w1, w2: (w1 * x_data + w2 - y_data) ** 2

w_start = np.array([0.0, 0.0])
grad_L = gradient(loss, w_start)
step_size = 0.1

gradient_angle_deg = np.degrees(np.arctan2(grad_L[1], grad_L[0])) % 360
steepest_descent_angle = (gradient_angle_deg + 180) % 360
print(f"Gradient at w_start: {np.round(grad_L, 3)}  (points toward {gradient_angle_deg:.1f} deg)")
print(f"Loss at w_start: {loss(*w_start):.4f}\n")

# Fine sweep (every 2 degrees) so we can actually locate the true minimizing direction,
# rather than getting lucky/unlucky with a handful of coarse candidate directions
sweep_deg = np.arange(0, 360, 2)
sweep_losses = []
for angle_deg in sweep_deg:
    theta = np.radians(angle_deg)
    direction = np.array([np.cos(theta), np.sin(theta)])
    sweep_losses.append(loss(*(w_start + step_size * direction)))
sweep_losses = np.array(sweep_losses)
best_angle = sweep_deg[np.argmin(sweep_losses)]

print(f"Best direction found by sweep: {best_angle} deg")
print(f"-gradient direction (predicted optimum): {steepest_descent_angle:.1f} deg")
print(f"Difference: {abs(best_angle - steepest_descent_angle):.1f} deg (small, limited by the 2-degree sweep resolution)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sweep_deg, sweep_losses, color="#4C72B0", lw=2)
ax.axvline(steepest_descent_angle, color="#C44E52", ls="--", label="-gradient direction (predicted optimum)")
ax.set_xlabel("step direction (degrees)"); ax.set_ylabel("loss after one step")
ax.set_title("Steepest descent direction minimizes loss after one step")
ax.legend()
plt.show()

## 🏋️ Exercises

### Warm-up
1. For $f(x,y) = x^2 - xy$ at $(2,1)$, compute the directional derivative toward
   $\hat{\mathbf{u}} = (1,1)/\sqrt{2}$ both directly from the limit definition and via
   $\nabla f \cdot \hat{\mathbf{u}}$. Confirm they match.

### Practice
2. For the bowl $f(x,y)=x^2+2y^2$ from the Visual Intuition cell, verify numerically that the
   directional derivative in the direction *perpendicular* to the gradient is exactly zero at
   3 different points of your choosing.

### Challenge
3. Modify the AI section's experiment to scan a full 360-degree sweep (every 1 degree, not
   just 8 directions) and plot loss-after-one-step as a continuous curve versus angle. Confirm
   the minimum of that curve occurs exactly at the $-\nabla L$ direction, then explain why, for
   a large step size $\eta$, this guarantee can break down (hint: think about what "steepest
   *locally*" means once you've moved far enough that the gradient itself has changed).

---

## 📚 Further Reading
- Deisenroth, Faisal & Ong, *Mathematics for Machine Learning*, Ch. 5.4–5.5 (Gradients, Directional Derivatives)
- Baydin et al., ["Automatic Differentiation in Machine Learning: a Survey"](https://arxiv.org/abs/1502.05767)

---

*Next notebook: [Jacobians and Hessians](06_jacobians_and_hessians.ipynb)*